# Modeling: Stent Thrombosis Prediction (VLST)

## Overview

This notebook trains and evaluates binary classifiers to predict **Stent thrombosis** using the preprocessed data from `../preprocessing/preprocessing.ipynb`. We use:

- **Data**: Preprocessed train/test arrays from `data/processed/` (resolved from repo root)
- **Models**: Logistic Regression, Decision Tree, Random Forest, Gaussian NB, CatBoost, XGBoost, LightGBM (with **GridSearchCV** where applicable)
- **Flowcharts**: Fitted GridSearchCV/pipeline objects are displayed so Jupyter shows the interactive sklearn pipeline diagram (expandable flowchart)
- **Imbalance**: Class weights in models, optional **SMOTE** on the training set (see §2), and metrics suited to imbalanced data (ROC-AUC, PR-AUC, F1, recall)
- **Artifacts**: Best models and results saved to `data/result/modeling/`

**Leakage comparison arm — ALL LEAKS ON.** This notebook is the *before* GridSearch. It keeps every leakage-prone signal identified in `baseline_plus_tabpfn.ipynb`: `Time since stent implantation`, `WBC`, raw lab recording precision (no quantization), and stent-brand collapse fit on the **full cohort** before the split. Twin: `baseline_without_tssi.ipynb` (full anti-leakage).

**Stored outputs in this file (Table S-TSSI “with TSSI”) are the previous processed-npy run.** They are kept so reports can still quote them until you re-execute. New runs write to `data/result/modeling_tssi_leakage/`.



## 1. Setup and Load Preprocessed Data

In [ ]:
import numpy as np
import pandas as pd
import os
import joblib
import warnings
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.base import BaseEstimator, ClassifierMixin
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from pathlib import Path
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    f1_score,
    recall_score,
    precision_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)

warnings.filterwarnings("ignore")
np.random.seed(42)


def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "data" / "raw" / "VLST.csv").is_file():
            return p
    if Path("/kaggle/working").is_dir():
        return Path("/kaggle/working")
    raise FileNotFoundError("Could not locate data/raw/VLST.csv above cwd.")


from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# ---------------------------------------------------------------------------
# LEAKY protocol. Keep every leakage-prone signal so this GridSearch is the
# *before* arm. The twin (baseline_without_tssi.ipynb) turns all of these off.
# Stored cell outputs / Table S-TSSI are from the previous npy run; they stay
# until you re-execute. New artifacts go to RESULT_DIR (new folder).
# ---------------------------------------------------------------------------
KEEP_TSSI = True
DROP_WBC = False
QUANTIZE_CLINICAL = False
STENT_ENCODER_TRAIN_ONLY = False
TEST_SIZE = 0.30
RANDOM_STATE = 42
TARGET = "Stent thrombosis"
ID_COLS = ["NO.", "Name"]
STENT_BRAND_COL = "Stent type-SES"
CLINICAL_QUANTIZE_PLACES = {"Cre": 0, "CaI": 2, "Fiberinogen": 1, "Fast-Glu": 1}

DROP_FEATURES = []
if not KEEP_TSSI:
    DROP_FEATURES.append("Time since stent implantation")
if DROP_WBC:
    DROP_FEATURES.append("WBC")


def _discover_vlst_csv(repo: Path) -> Path:
    env = os.environ.get("VLST_RAW_CSV")
    if env and Path(env).is_file():
        return Path(env)
    local = repo / "data" / "raw" / "VLST.csv"
    if local.is_file():
        return local
    for p in (
        Path("/kaggle/input/datasets/amirmahdidaraei/vlst-data/VLST.csv"),
        Path("/kaggle/input/vlst-data/VLST.csv"),
        Path("/kaggle/input/datasets/amirinho661/vlst-figshare-7409606/VLST.csv"),
    ):
        if p.is_file():
            return p
    base = Path("/kaggle/input")
    if base.is_dir():
        for p in base.rglob("VLST.csv"):
            if p.is_file():
                return p
    raise FileNotFoundError("VLST.csv not found.")


def quantize_clinical_precision(frame, places_by_column):
    applied = {}
    for column, places in places_by_column.items():
        if column not in frame.columns or not pd.api.types.is_numeric_dtype(frame[column]):
            continue
        before = pd.to_numeric(frame[column], errors="coerce")
        rounded = before.round(int(places))
        frame[column] = rounded
        applied[column] = {"places": int(places), "n_changed": int((before != rounded).fillna(False).sum())}
    return applied


def _load_stent_encoding():
    import re
    import sys

    search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for extra in (Path("/kaggle/working"), Path("/kaggle/input"), Path("/kaggle/usr/lib")):
        if extra.is_dir():
            search_roots.append(extra)
            search_roots.extend(extra.glob("*"))
    for d in search_roots:
        for tools in (d / "code" / "modeling" / "tools", d / "modeling" / "tools", d):
            if (Path(tools) / "stent_encoding.py").is_file():
                tools = Path(tools)
                if str(tools) not in sys.path:
                    sys.path.insert(0, str(tools))
                try:
                    from stent_encoding import (
                        coerce_stent_class_flags,
                        fit_stent_brand_encoder,
                        transform_stent_brand_column,
                    )
                except ImportError:
                    continue
                print("Stent encoder: imported from", tools / "stent_encoding.py")
                return coerce_stent_class_flags, fit_stent_brand_encoder, transform_stent_brand_column

    print("Stent encoder: using in-notebook fallback.")
    STENT_BRAND_MIN_COUNT = 30
    _BRAND_ALIASES = {"xiencex": "xiencev", "resolut": "resolute", "parnter": "partner", "endeavor": "endeavor", "cypher": "cypher"}
    STENT_CLASS_FLAG_COLS = ("PES", "ZES", "EVS")

    def canonicalize_stent_brand(value):
        if pd.isna(value) or str(value).strip() == "":
            return "missing"
        s = str(value).strip().replace("：", ":").lower()
        s = re.sub(r"\s+", "", s)
        if ":" in s:
            s = s.split(":")[-1]
        for sep in ("，", ",", "/"):
            if sep in s:
                s = s.split(sep)[0]
        return _BRAND_ALIASES.get(s, s)

    def coerce_stent_class_flags(df):
        out = df
        for col in STENT_CLASS_FLAG_COLS:
            if col not in out.columns:
                continue
            s = pd.to_numeric(out[col], errors="coerce").fillna(0).astype(int)
            bad = ~s.isin([0, 1])
            if bad.any():
                raise ValueError(f"{col}: expected 0/1, got {out.loc[bad, col].unique()[:10]}")
            out[col] = s
        return out

    def fit_stent_brand_encoder(series, min_count=STENT_BRAND_MIN_COUNT):
        meta = {"min_count": min_count, "numeric": False, "kept": set(), "n_raw": 0, "n_levels": 0, "applied": False}
        if pd.api.types.is_numeric_dtype(series):
            n = int(series.nunique(dropna=True))
            meta.update({"numeric": True, "n_raw": n, "n_levels": n})
            return meta
        canon = series.map(canonicalize_stent_brand)
        counts = canon.value_counts()
        kept = set(counts[counts >= min_count].index)
        meta.update({
            "kept": kept,
            "n_raw": int(series.nunique(dropna=True)),
            "n_levels": int(len(kept) + int((counts < min_count).any())),
            "applied": True,
        })
        return meta

    def transform_stent_brand_column(frame, codebook, raw_col=STENT_BRAND_COL, inplace=False):
        out = frame if inplace else frame.copy()
        if raw_col not in out.columns or codebook.get("numeric") or not codebook.get("applied"):
            return out
        canon = out[raw_col].map(canonicalize_stent_brand)
        kept = codebook["kept"]
        out[raw_col] = canon.where(canon.isin(kept), "other").astype("object")
        return out

    return coerce_stent_class_flags, fit_stent_brand_encoder, transform_stent_brand_column


def encode_stent_on_fold(X_tr, X_va, raw_col=STENT_BRAND_COL, min_count=30):
    if raw_col not in X_tr.columns:
        return X_tr, X_va, {"applied": False, "n_raw": 0, "n_levels": 0, "min_count": min_count}
    codebook = fit_stent_brand_encoder(X_tr[raw_col], min_count=min_count)
    return (
        transform_stent_brand_column(X_tr, codebook, raw_col=raw_col),
        transform_stent_brand_column(X_va, codebook, raw_col=raw_col),
        codebook,
    )


is_kaggle_env = Path("/kaggle/working").is_dir()
REPO_ROOT = _find_repo_root()
RAW_PATH = _discover_vlst_csv(REPO_ROOT)
RESULT_DIR = str(
    (Path("/kaggle/working") if is_kaggle_env else REPO_ROOT / "data" / "result")
    / "modeling_tssi_leakage"
)
os.makedirs(RESULT_DIR, exist_ok=True)

coerce_stent_class_flags, fit_stent_brand_encoder, transform_stent_brand_column = _load_stent_encoding()

df = pd.read_csv(RAW_PATH)
df = coerce_stent_class_flags(df)
if QUANTIZE_CLINICAL:
    _q = quantize_clinical_precision(df, CLINICAL_QUANTIZE_PLACES)
    print(
        "Clinical quantization (file-level, before split):",
        ", ".join(f"{c}->{m['places']}dp ({m['n_changed']} changed)" for c, m in _q.items()) or "(none)",
    )
drop_cols = [c for c in ID_COLS + DROP_FEATURES if c in df.columns]
print("Dropped:", drop_cols)
X_df = df.drop(columns=drop_cols + [TARGET], errors="ignore")
y = df[TARGET].astype(int)
if set(y.unique()) != {0, 1}:
    raise ValueError(f"{TARGET!r} must be binary 0/1.")

print(
    f"KEEP_TSSI={KEEP_TSSI} | DROP_WBC={DROP_WBC} | quantize={QUANTIZE_CLINICAL} | "
    f"stent_train_only={STENT_ENCODER_TRAIN_ONLY}"
)
print(f"RAW_PATH: {RAW_PATH}")
print(f"RESULT_DIR (new run): {RESULT_DIR}")
print(
    "Frozen Table S-TSSI (pre-anti-leakage stored npy run; do not overwrite reports until re-run): "
    "see rebuild_tssi_leakage_table.py ROWS."
)

X_train_df, X_test_df, y_train, y_test = train_test_split(
    X_df, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
X_train_df = X_train_df.copy()
X_test_df = X_test_df.copy()
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()

if STENT_ENCODER_TRAIN_ONLY:
    X_train_df, X_test_df, stent_meta = encode_stent_on_fold(X_train_df, X_test_df)
    if stent_meta.get("applied"):
        print(
            f"Stent brand (train only): {stent_meta.get('n_raw')} raw -> "
            f"{stent_meta.get('n_levels')} levels (min_count={stent_meta.get('min_count')})"
        )
else:
    codebook = fit_stent_brand_encoder(X_df[STENT_BRAND_COL]) if STENT_BRAND_COL in X_df.columns else {}
    X_train_df = transform_stent_brand_column(X_train_df, codebook)
    X_test_df = transform_stent_brand_column(X_test_df, codebook)

numeric_features = X_train_df.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in X_train_df.columns if c not in numeric_features]
preprocessor = ColumnTransformer(
    transformers=[
        ("num", SkPipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_features),
        ("cat", SkPipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical_features),
    ]
)
# Fit on the training split only — GridSearch inner CV still sees a frozen
# train-fit transform (same contract as the old preprocessing.ipynb npy).
preprocessor.fit(X_train_df)
X_train = np.asarray(preprocessor.transform(X_train_df), dtype=float)
X_test = np.asarray(preprocessor.transform(X_test_df), dtype=float)
feature_names = preprocessor.get_feature_names_out().tolist()

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Features: {len(feature_names)}")
print(f"Train target: 0={np.sum(y_train == 0)}, 1={np.sum(y_train == 1)}")
print(f"Test target:  0={np.sum(y_test == 0)}, 1={np.sum(y_test == 1)}")
tssi_present = any("Time since stent implantation" in n for n in feature_names)
wbc_present = any(n.endswith("WBC") or n == "WBC" or n.endswith("__WBC") for n in feature_names)
print(f"TSSI in model matrix: {tssi_present} | WBC in model matrix: {wbc_present}")


## 2a. Addressing Class Imbalance (Optional SMOTE)

As noted in **preprocessing.ipynb**, the target is strongly imbalanced. We use **class weights** in the models below and optionally apply **SMOTE** to the **training set only** (test set unchanged). Set `USE_SMOTE = False` to use the original training set.

In [ ]:
from imblearn.over_sampling import SMOTE

USE_SMOTE = (
    True  # Set to False to train on original (imbalanced) data with class weights only
)

if USE_SMOTE:
    n_minority = (y_train == 1).sum()
    k = min(5, n_minority - 1) if n_minority > 1 else 1
    if k < 1:
        print("SMOTE skipped: minority class has too few samples.")
    else:
        smote = SMOTE(random_state=42, k_neighbors=k)
        X_train, y_train = smote.fit_resample(X_train, y_train)
        print(
            f"SMOTE applied: train size -> {X_train.shape[0]} (0={np.sum(y_train == 0)}, 1={np.sum(y_train == 1)})"
        )
else:
    print("Using original (imbalanced) training set with class weights in models.")

## 2. Cross-Validation and Evaluation Helpers

We use stratified 5-fold CV in GridSearchCV and define a small helper to compute metrics and optionally plot ROC/PR curves.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def evaluate_model(name, model, X_te=X_test, y_te=y_test):
    """Print metrics and return dict of test metrics."""
    y_pred = model.predict(X_te)
    y_prob = (
        model.predict_proba(X_te)[:, 1] if hasattr(model, "predict_proba") else None
    )
    out = {
        "model": name,
        "accuracy": (y_pred == y_te).mean(),
        "f1": f1_score(y_te, y_pred, zero_division=0),
        "recall": recall_score(y_te, y_pred, zero_division=0),
        "precision": precision_score(y_te, y_pred, zero_division=0),
    }
    if y_prob is not None:
        out["roc_auc"] = roc_auc_score(y_te, y_prob)
        out["pr_auc"] = average_precision_score(y_te, y_prob)
    print(classification_report(y_te, y_pred))
    print("ROC-AUC:", out.get("roc_auc"), "| PR-AUC:", out.get("pr_auc"))
    return out

# Scout metric for this 70/30 GridSearch. Winners are NOT passed into
# baseline_plus_tabpfn.ipynb (nested CV uses defaults + threshold search).
# Stored Table S-TSSI used scoring="f1". average_precision matches Part 4 ranking.
GRID_SCORING = "average_precision"


## 3. Logistic Regression + GridSearchCV

We tune regularization (C, penalty) and use **class_weight='balanced'** for the imbalanced target. After fitting, displaying the GridSearchCV object shows the pipeline flowchart in Jupyter.

In [ ]:
param_grid_lr = [
    {"C": [0.01, 0.1, 1.0, 10.0, 100.0], "penalty": ["l2"], "solver": ["lbfgs"], "max_iter": [2000]},
    {"C": [0.01, 0.1, 1.0, 10.0, 100.0], "penalty": ["l1"], "solver": ["liblinear"], "max_iter": [2000]},
]
grid_lr = GridSearchCV(
    estimator=LogisticRegression(class_weight="balanced", random_state=42),
    param_grid=param_grid_lr,
    scoring=GRID_SCORING,
    cv=cv,
    n_jobs=-1,
    verbose=0,
)

grid_lr.fit(X_train, y_train)
print("Best params:", grid_lr.best_params_)
print("Best CV", GRID_SCORING, ":", round(grid_lr.best_score_, 4))


In [ ]:
# Display fitted GridSearchCV (flowchart of the best pipeline in Jupyter)
display(grid_lr)

In [ ]:
best_lr = grid_lr.best_estimator_
results_list = [evaluate_model("LogisticRegression", best_lr)]

### 3.1 Logistic Regression – CV results (top 5 by F1)

View GridSearchCV `cv_results_` to see how different hyperparameters performed.

In [ ]:
cv_res_lr = pd.DataFrame(grid_lr.cv_results_)
cols = [
    c for c in cv_res_lr.columns if c.startswith("param_") or c == "mean_test_score"
]
display(cv_res_lr[cols].sort_values("mean_test_score", ascending=False).head())

## 4. Decision Tree + GridSearchCV

Tune tree depth, min_samples_split, min_samples_leaf, and criterion. After fitting, display the grid to see the pipeline flowchart.

In [ ]:
param_grid_dt = {
    "max_depth": [3, 5, 7, 10, 15, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "criterion": ["gini", "entropy"],
    "max_features": [None, "sqrt"],
}
grid_dt = GridSearchCV(
    estimator=DecisionTreeClassifier(class_weight="balanced", random_state=42),
    param_grid=param_grid_dt,
    scoring=GRID_SCORING,
    cv=cv,
    n_jobs=-1,
    verbose=0,
)

grid_dt.fit(X_train, y_train)
print("Best params:", grid_dt.best_params_)
print("Best CV", GRID_SCORING, ":", round(grid_dt.best_score_, 4))


In [ ]:
# Display fitted GridSearchCV (flowchart)
display(grid_dt)

In [ ]:
best_dt = grid_dt.best_estimator_
results_list.append(evaluate_model("DecisionTree", best_dt))

### 4.1 Decision Tree: Visualize the Fitted Tree

Plot the best decision tree (limited depth for readability). Feature names from preprocessing are used for labels.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(20, 12))
plot_tree(
    best_dt,
    max_depth=8,
    feature_names=feature_names,
    class_names=["No thrombosis", "Thrombosis"],
    filled=True,
    rounded=True,
    ax=ax,
    fontsize=8,
)
ax.set_title("Decision Tree (best from GridSearchCV, max_depth=4 for readability)")
plt.tight_layout()
plt.savefig(
    os.path.join(RESULT_DIR, "decision_tree_plot.png"), dpi=150, bbox_inches="tight"
)
plt.show()
print("Saved: decision_tree_plot.png")

## 5. Random Forest + GridSearchCV

Tune n_estimators, max_depth, and min_samples_leaf. Display the fitted grid for the pipeline flowchart.

In [ ]:
param_grid_rf = {
    "n_estimators": [200, 400, 800],
    "max_depth": [5, 10, 20, None],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt", 0.5],
}
grid_rf = GridSearchCV(
    estimator=RandomForestClassifier(class_weight="balanced", random_state=42),
    param_grid=param_grid_rf,
    scoring=GRID_SCORING,
    cv=cv,
    n_jobs=-1,
    verbose=0,
)

grid_rf.fit(X_train, y_train)
print("Best params:", grid_rf.best_params_)
print("Best CV", GRID_SCORING, ":", round(grid_rf.best_score_, 4))


In [ ]:
# Display fitted GridSearchCV (flowchart)
display(grid_rf)

In [ ]:
best_rf = grid_rf.best_estimator_
results_list.append(evaluate_model("RandomForest", best_rf))

## 5b. Gaussian Naive Bayes + GridSearchCV

Gaussian NB is fast and works well with balanced or moderately imbalanced data. We tune `var_smoothing` for stability.

In [ ]:
param_grid_gnb = {"var_smoothing": np.logspace(-12, -6, 13)}
grid_gnb = GridSearchCV(
    estimator=GaussianNB(),
    param_grid=param_grid_gnb,
    scoring=GRID_SCORING,
    cv=cv,
    n_jobs=-1,
    verbose=0,
)
grid_gnb.fit(X_train, y_train)
print("Best params:", grid_gnb.best_params_)
print("Best CV", GRID_SCORING, ":", round(grid_gnb.best_score_, 4))


In [ ]:
display(grid_gnb)

In [ ]:
best_gnb = grid_gnb.best_estimator_
results_list.append(evaluate_model("GaussianNB", best_gnb))

## 5c. CatBoost + GridSearchCV

CatBoost handles categorical features natively; here we use the preprocessed numeric matrix.

In [ ]:
# Wrapper so CatBoost works with GridSearchCV
class CatBoostClassifierWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, **kwargs):
        self.catboost_ = CatBoostClassifier(**kwargs)

    def get_params(self, deep=True):
        return self.catboost_.get_params(deep=deep)

    def set_params(self, **params):
        self.catboost_.set_params(**params)
        return self

    def fit(self, X, y, **fit_params):
        self.catboost_.fit(X, y, **fit_params)
        return self

    def predict(self, X):
        return self.catboost_.predict(X)

    def predict_proba(self, X):
        return self.catboost_.predict_proba(X)

In [ ]:
param_grid_cat = {
    "depth": [4, 6, 8],
    "learning_rate": [0.03, 0.05, 0.1],
    "iterations": [200, 400],
    "l2_leaf_reg": [1, 3, 5],
}
grid_cat = GridSearchCV(
    estimator=CatBoostClassifierWrapper(
        random_state=42,
        verbose=0,
        auto_class_weights="Balanced",
        eval_metric="PRAUC",
    ),
    param_grid=param_grid_cat,
    scoring=GRID_SCORING,
    cv=cv,
    n_jobs=-1,
    verbose=0,
)
grid_cat.fit(X_train, y_train)
print("Best params:", grid_cat.best_params_)
print("Best CV", GRID_SCORING, ":", round(grid_cat.best_score_, 4))
best_cat = grid_cat.best_estimator_
display(grid_cat)
results_list.append(evaluate_model("CatBoost", best_cat))


## 5d. XGBoost + GridSearchCV

Gradient boosting; often strong on tabular data.

In [ ]:
scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
param_grid_xgb = {
    "max_depth": [3, 5, 7],
    "learning_rate": [0.03, 0.05, 0.1],
    "n_estimators": [200, 400],
    "min_child_weight": [1, 3],
    "subsample": [0.8, 1.0],
}
grid_xgb = GridSearchCV(
    estimator=XGBClassifier(
        random_state=42,
        use_label_encoder=False,
        eval_metric="aucpr",
        scale_pos_weight=scale_pos_weight,
    ),
    param_grid=param_grid_xgb,
    scoring=GRID_SCORING,
    cv=cv,
    n_jobs=-1,
    verbose=0,
)
grid_xgb.fit(X_train, y_train)
print("Best params:", grid_xgb.best_params_)
print("Best CV", GRID_SCORING, ":", round(grid_xgb.best_score_, 4))
best_xgb = grid_xgb.best_estimator_
display(grid_xgb)
results_list.append(evaluate_model("XGBoost", best_xgb))


## 5e. LightGBM + GridSearchCV

LightGBM is fast and often performs well on tabular data.

In [ ]:
param_grid_lgb = {
    "max_depth": [3, 5, 7],
    "learning_rate": [0.03, 0.05, 0.1],
    "n_estimators": [200, 400],
    "min_child_samples": [10, 20, 40],
    "num_leaves": [15, 31],
}
grid_lgb = GridSearchCV(
    estimator=LGBMClassifier(
        random_state=42,
        class_weight="balanced",
        metric="average_precision",
        verbosity=-1,
    ),
    param_grid=param_grid_lgb,
    scoring=GRID_SCORING,
    cv=cv,
    n_jobs=-1,
    verbose=0,
)
grid_lgb.fit(X_train, y_train)
print("Best params:", grid_lgb.best_params_)
print("Best CV", GRID_SCORING, ":", round(grid_lgb.best_score_, 4))
best_lgb = grid_lgb.best_estimator_
display(grid_lgb)
results_list.append(evaluate_model("LightGBM", best_lgb))


## 6. ROC and Precision-Recall Curves

Compare all three models on ROC and PR curves (PR is more informative for imbalanced data).

In [ ]:
curve_models = [
    ("LR", best_lr),
    ("DT", best_dt),
    ("RF", best_rf),
    ("GNB", best_gnb),
    ("CatBoost", best_cat),
    ("XGB", best_xgb),
    ("LGB", best_lgb),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, model in curve_models:
    y_prob = model.predict_proba(X_test)[:, 1]
    RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[0], name=name)
axes[0].set_title("ROC curves")
axes[0].legend()

for name, model in curve_models:
    y_prob = model.predict_proba(X_test)[:, 1]
    PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=axes[1], name=name)
axes[1].set_title("Precision-Recall curves")
axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, "roc_pr_curves.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: roc_pr_curves.png")

## 7. Confusion Matrices

Heatmaps of confusion matrices for each model on the test set.

In [ ]:
import seaborn as sns

cm_models = [
    ("LogReg", best_lr),
    ("DT", best_dt),
    ("RF", best_rf),
    ("GNB", best_gnb),
    ("CatBoost", best_cat),
    ("XGB", best_xgb),
    ("LGB", best_lgb),
]
n_models = len(cm_models)
n_cols = 3
n_rows = (n_models + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
axes = axes.flatten() if n_models > 1 else [axes]
for ax, (name, model) in zip(axes, cm_models):
    cm = confusion_matrix(y_test, model.predict(X_test))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        ax=ax,
        cmap="Blues",
        xticklabels=["No", "Yes"],
        yticklabels=["No", "Yes"],
    )
    ax.set_title(name)
    ax.set_ylabel("True")
    ax.set_xlabel("Predicted")
for ax in axes[len(cm_models) :]:
    ax.axis("off")
plt.tight_layout()
plt.savefig(
    os.path.join(RESULT_DIR, "confusion_matrices.png"), dpi=150, bbox_inches="tight"
)
plt.show()
print("Saved: confusion_matrices.png")

## 8. Threshold Analysis (Optional)

For the best model by F1, we can sweep the decision threshold and report precision, recall, and F1. This helps choose a threshold if you want to favor recall over precision or vice versa.

In [ ]:
# Pick best model by test F1 for threshold analysis
threshold_candidates = [
    ("LR", best_lr),
    ("DT", best_dt),
    ("RF", best_rf),
    ("GNB", best_gnb),
    ("CatBoost", best_cat),
    ("XGB", best_xgb),
    ("LGB", best_lgb),
]
best_by_f1 = max(
    threshold_candidates, key=lambda t: f1_score(y_test, t[1].predict(X_test))
)
name_best, model_best = best_by_f1
y_prob_best = model_best.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.1, 0.95, 0.05)
rows = []
for t in thresholds:
    y_pt = (y_prob_best >= t).astype(int)
    rows.append(
        {
            "threshold": t,
            "precision": precision_score(y_test, y_pt, zero_division=0),
            "recall": recall_score(y_test, y_pt, zero_division=0),
            "f1": f1_score(y_test, y_pt, zero_division=0),
        }
    )
threshold_df = pd.DataFrame(rows)
print(f"Threshold analysis for best model ({name_best}):")
display(threshold_df)

## 9. Results Summary and Saving Artifacts

Summarize test metrics for all models and save best estimators and results table.

In [ ]:
results_df = pd.DataFrame(results_list)
results_df = results_df.reindex(
    columns=["model", "accuracy", "f1", "recall", "precision", "roc_auc", "pr_auc"]
)
print("Test set metrics:")
display(results_df)

results_df.to_csv(os.path.join(RESULT_DIR, "test_metrics.csv"), index=False)
threshold_df.to_csv(os.path.join(RESULT_DIR, "threshold_analysis.csv"), index=False)

joblib.dump(best_lr, os.path.join(RESULT_DIR, "best_logistic.joblib"))
joblib.dump(best_dt, os.path.join(RESULT_DIR, "best_decision_tree.joblib"))
joblib.dump(best_rf, os.path.join(RESULT_DIR, "best_random_forest.joblib"))
joblib.dump(best_gnb, os.path.join(RESULT_DIR, "best_gaussian_nb.joblib"))
joblib.dump(best_cat, os.path.join(RESULT_DIR, "best_catboost.joblib"))
joblib.dump(best_xgb, os.path.join(RESULT_DIR, "best_xgboost.joblib"))
joblib.dump(best_lgb, os.path.join(RESULT_DIR, "best_lightgbm.joblib"))

print("\nSaved to", RESULT_DIR)
for f in sorted(os.listdir(RESULT_DIR)):
    print(" ", f)

## 10. Summary

- **Data**: Loaded from `data/processed/` (`../preprocessing/preprocessing.ipynb`).
- **Models**: Logistic Regression, Decision Tree, Random Forest, Gaussian NB, CatBoost, XGBoost, LightGBM (with **GridSearchCV** where applicable); each fitted grid is **displayed** so Jupyter shows the pipeline flowchart.
- **Imbalance**: `class_weight='balanced'` (and scale_pos_weight / auto_class_weights where supported), optional SMOTE on training data, and metrics: F1, ROC-AUC, PR-AUC, recall.
- **Visuals**: Decision tree plot, ROC/PR curves, confusion matrices, threshold table.
- **Artifacts**: Best estimators and metrics saved under `data/result/modeling/`.

For inference on new data: preprocess with `preprocessor.joblib` and `scaler.joblib` from preprocessing, then call `best_*.predict(X)` or `best_*.predict_proba(X)`.

**This code (not yet the stored outputs):** raw `VLST.csv` with **all leaks left in** (TSSI, WBC, unquantized labs, full-cohort stent collapse). Re-run to refresh metrics. Previous Table S-TSSI numbers remain in the executed outputs until then.


# Why are the model metrics so incredibly good and high (Why are the models overly accurate? Is there data leakage?)

Here’s what the code and data show.

## 1. The pipeline is not “leaking” in the usual ML sense

From `preprocessing.ipynb`:

- `train_test_split` is stratified, then **only** `X_train` is used to `fit` the `ColumnTransformer` and `StandardScaler`; `X_test` is only `transform`ed.
- `y_train` / `y_test` are aligned with the same split.

So there is **no** bug like: scaling on full data, fitting on test, or shuffled rows between `X` and `y`.

`evaluate_model` uses `X_test` and `y_test` consistently. The CatBoost wrapper’s `fit` / `predict` / `predict_proba` wiring is normal.

---

## 2. Why metrics look “impossible” anyway: **feature–label leakage**, not a code bug

The problem is **what** “Time since stent implantation” means for **class 0 vs class 1** in this file.

On the full raw data:

| Class | `Time since stent implantation` |
|--------|----------------------------------|
| **0 (no thrombosis)** | **min = 1241**, max = 1605 (all values ≥ 1241) |
| **1 (thrombosis)** | **min = 380**, includes many values **&lt; 1241** |

So:

- **Every** control has time ≥ **1241** days.
- **Many** cases have time **&lt; 1241** days.

A single rule **“if time &lt; 1241 → predict thrombosis”** already gives:

- **0** false positives among controls (no control has time &lt; 1241),
- **~99.6% accuracy** on the full dataset,
- and a **very high** ROC-AUC (because time almost separates the groups).

That matches what you see with CatBoost: **0.9999 CV F1**, **~1.0 ROC-AUC**, **precision 1.0 for class 0** on the test set — the model can rely on (or amplify) this signal.

**Interpretation (substance, not sklearn):**  
For **non-events**, “time since stent implantation” behaves like a **narrow follow-up window** (everyone without an event lands in a high band). For **events**, it behaves like **time from implant to thrombosis**, which can be shorter. So the column is **not** a clean “baseline predictor of VLST”; it **mixes different time definitions** for cases vs controls. That is **structural / temporal leakage**: the label is partly encoded in how the variable is defined.

Univariate checks also show **“Time since stent implantation”** as by far the strongest signal (e.g. very high univariate AUC on a test split). So the inflated metrics are **not** mysterious once you look at this column.

---

## 3. What to do for a **fair** prediction task

1. **Drop** `Time since stent implantation` (and any other variables that are **only comparable** if defined the same way for cases and controls), then re-run preprocessing and modeling.
2. Or **redefine** the variable so it is **time from implant to a common landmark** (e.g. index procedure date, or end of follow-up) for **both** groups, with the same definition for everyone.
3. If the scientific question is **when** thrombosis occurs, use **survival / time-to-event** methods instead of a single binary classifier with a time that is not a pure risk factor.

---

**Bottom line:** There is **no evidence of a bug** in the train/test or preprocessing code that would explain 0.9999. The explanation is **data leakage through `Time since stent implantation`** (and possibly other time-related fields), not incorrect sklearn usage. Dropping or fixing that variable will bring metrics down to something more interpretable for real risk prediction.